# Load library

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl
import anndata as ann
import random, os
from scipy.stats import pearsonr as pr
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score as f1
from sklearn.metrics import precision_recall_curve as prc
from sklearn.metrics import silhouette_score as sil
from sklearn.metrics import auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.metrics import silhouette_score
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
import psutil
import os, sys
import gc
import scipy.sparse as sp
from harmony import harmonize
from tqdm import tqdm
import h5py

In [2]:
sc.set_figure_params(dpi=200)

# General processing functinos

In [3]:
def whats_memory_eater():
    # Build reverse map of object id -> variable name from globals
    name_map = {id(obj): name for name, obj in globals().items()}

    # Get all tracked objects
    all_objects = gc.get_objects()

    # Safely get size and match variable name
    sizes = []
    for obj in all_objects:
        try:
            size = sys.getsizeof(obj)
            obj_id = id(obj)
            name = name_map.get(obj_id, None)
            sizes.append((size, type(obj), name, repr(obj)[:100]))
        except Exception:
            continue

    # Sort and print top 10
    sizes.sort(reverse=True, key=lambda x: x[0])

    for size, obj_type, name, preview in sizes[:10]:
        print(f"Size: {size / 1024**3} GB | Type: {obj_type} | Name: {name} | Object: {preview}")


In [4]:
def memory_usgae():
    gc.collect()
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024**3  # in GB

    print(f"Current memory usage: {memory_gb:.2f} GB")

In [5]:
def pca_and_umap(adata):
    sc.tl.pca(adata, svd_solver="arpack")
    # sc.pl.pca(ad, color='source')
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)

In [6]:
def load_and_preprocess_project(base_path, Project_ID, metadata_idx_key='Cell', 
                                Primary_or_Metastatic = 'Primary', remove_doublets = True, doublet_rate=0.06,
                                further_pre = False, file_prefix= None):
    """
    Load and preprocess a single scRNA-seq project with standard filtering and UMAP.
    
    Assumes the base_path contains:
        - One .mtx file (count matrix)
        - One barcodes.csv
        - One features.csv
        - One meta_all.csv
    """
    # Automatically detect files
    files = os.listdir(base_path)
    metadata_file = None
    
    if file_prefix == None:

        mtx_file = [os.path.join(base_path, f) for f in files if f.endswith('.mtx')][0]
        print(mtx_file)
        barcodes_file = [os.path.join(base_path, f) for f in files if 'barcode' in f][0]
        print(barcodes_file)
        try:
            features_file = [os.path.join(base_path, f) for f in files if 'feature' in f][0]
        except:
            features_file = [os.path.join(base_path, f) for f in files if 'genes' in f.lower()][0]
        print(features_file)
        metadata_file = [os.path.join(base_path, f) for f in files if 'meta' in f][0]
        print(metadata_file)
    else:
        for f in files:
            if not f.startswith(file_prefix):
                continue
            if f.endswith('mtx'):
                mtx_file = os.path.join(base_path, f)
                print(mtx_file)
            elif 'barcode' in f:
                barcodes_file = os.path.join(base_path, f)
                print(barcodes_file)
            elif 'feature' in f:
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'genes' in f.lower():
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'meta' in f:
                metadata_file = os.path.join(base_path, f)
                print(metadata_file)
            else:
                continue
    # print(metadata_file)
    print(f"Loading: {mtx_file}")

    # Load matrix
    adata = sc.read_mtx(mtx_file)
    adata = adata.transpose()  # Important: make cells as rows, genes as columns

    # Load barcodes and features
    if barcodes_file.endswith('tsv'):
        barcodes = pd.read_csv(barcodes_file, sep='\t', header=None)  # no header=None here
    else:
        barcodes = pd.read_csv(barcodes_file)  # no header=None here
    display(barcodes)
    
    if features_file.endswith('tsv'):
        genes = pd.read_csv(features_file, sep='\t', header=None)  # no header=None here
    else:
        genes = pd.read_csv(features_file)  # no header=None here
    display(genes)

    # Assign barcodes and gene names (convert to string)
    if barcodes.shape[1] > 1:
        adata.obs_names = barcodes.iloc[:, 1].astype(str).values
    else:
        adata.obs_names = barcodes.iloc[:, 0].astype(str).values
    
    if genes.shape[1] > 1:
        adata.var_names = genes.iloc[:, 1].astype(str).values
    else:
        adata.var_names = genes.iloc[:, 0].astype(str).values
    # adata.var_names = genes.iloc[:, 0].astype(str).values
    display(adata.to_df())

    # Load and merge metadata
    # if metadata_file
    try:
        if metadata_file.endswith('tsv'):
            metadata = pd.read_csv(metadata_file, sep='\t')
        elif metadata_file.endswith('csv'):
            metadata = pd.read_csv(metadata_file)
        metadata.index = metadata[metadata_idx_key]
        adata.obs = adata.obs.join(metadata, how='left')
    except:
        pass         
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')

    # Calculate QC metrics
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    # Standard cell filtering
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) & 
                  (adata.obs['n_genes_by_counts'] <= 5000) & 
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    # Normalize and log transform
    adata.raw = adata.copy()
    
    if further_pre:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # Highly variable genes
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

        # Keep only HVGs
        # adata = adata[:, adata.var.highly_variable]

        # Scale
        sc.pp.scale(adata, max_value=10)

        # PCA
        sc.tl.pca(adata, svd_solver='arpack')

        # Neighbors
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)

        # UMAP
        sc.tl.umap(adata)

    print(f"Finished processing {Project_ID}. Shape: {adata.shape}")

    return adata


In [7]:
def filter_and_recompute(adata, celltype_col, celltypes_to_keep, further_pre = False):
    """
    Filters an AnnData object to keep only specified cell types, 
    then recalculates PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object
    - celltype_col: str, the column in adata.obs containing cell type annotations
    - celltypes_to_keep: list of str, the cell types you want to keep

    Returns:
    - filtered and recalculated AnnData object
    """
    # Step 1: Filter cells
    print(f"Original shape: {adata.shape}")
    adata_filtered = adata[adata.obs[celltype_col].isin(set(celltypes_to_keep))].copy()
    print(f"Filtered shape: {adata_filtered.shape}")

    # Step 2: Recalculate PCA and UMAP
    # (Assumes data is already normalized and scaled)
    if further_pre:
        sc.tl.pca(adata_filtered, svd_solver='arpack')
        sc.pp.neighbors(adata_filtered, n_neighbors=15, n_pcs=40)
        sc.tl.umap(adata_filtered)

        print("Recalculated PCA and UMAP.")
    return adata_filtered

In [8]:
def reprocess_from_raw_layer(adata, Project_ID, Primary_or_Metastatic='Primary', 
                             further_pre=False, remove_doublets=True, doublet_rate=0.06):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes doublet removal, normalization, HVG selection, PCA, neighbors, and UMAP.
    
    Parameters:
    - adata: AnnData object, must have .raw set
    - Project_ID: Project identifier
    - Primary_or_Metastatic: Sample type ('Primary' or 'Metastatic')
    - further_pre: Whether to do full preprocessing (normalization, PCA, UMAP)
    - remove_doublets: Whether to run doublet detection and filtering
    - doublet_rate: Expected doublet rate (default 0.06 = 6%)
    
    Returns:
    - Processed AnnData object (modifies in place)
    """
    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")
    
    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')
    
    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # Add metadata
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        print('Normalizing...')
        # Normalize and log transform
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        
        print('Scaling...')
        sc.pp.scale(adata, max_value=10)
        
        print('Computing PCA...')
        sc.tl.pca(adata, svd_solver='arpack')
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)
    
    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata

In [9]:
def reprocess_all(adata, further_pre = True):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes normalization, HVG selection, PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object, must have .raw set

    Returns:
    - Processed AnnData object (modifies in place)
    """

    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")

    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()

    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # adata.obs['Project_ID'] = Project_ID
    # adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        # Normalize and log transform
        print('Normalizing...')
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # HVG selection
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
        # adata = adata[:, adata.var.highly_variable]
        '''
        sc.pp.highly_variable_genes(
            adata,
            flavor="seurat_v3",  # best for batch-aware HVG selection
            n_top_genes=2000,
            batch_key="Final_sample_id"  # or whatever your batch label column is
        )
        '''
        
        # adata = adata[:, adata.var.highly_variable].copy()

        # Scale
        # print('Scaling...')
        # sc.pp.scale(adata, max_value=10)
        print('Computing PCA...')
        # PCA, neighbors, UMAP
        sc.tl.pca(adata, zero_center=False)
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)

    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata


In [10]:
def reset_plot():
    # After running scrublet, reset the display settings
    import matplotlib.pyplot as plt
    import matplotlib
    %matplotlib inline

    # Reset matplotlib backend
    matplotlib.use('module://matplotlib_inline.backend_inline')

    # Reset scanpy settings
    import scanpy as sc
    sc.settings.autoshow = True
    sc.settings.set_figure_params(dpi=100, facecolor='white')

# Ovarian Cancer(OVC)

## A pan-cancer blueprint of the heterogeneous tumor microenvironment revealed by single-cell profiling


Paper: https://www.nature.com/articles/s41422-020-0355-0#Fig3

Data downloaded from: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0

Link: 
- Matrix: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0 (Ovarian cancer - Counts Matrix)
- Patient metadata: https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM13_ESM.pdf
- Sequecing quality: https://www.nature.com/
https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM14_ESM.pdf

In [10]:
# Set the project directory
project_dir = "../../Data/OVC/2100-Ovariancancer/"  # <- change this for each project

# Load and preprocess
ad = load_and_preprocess_project(project_dir, 
                                 Project_ID='2100-Ovariancancer', 
                                 Primary_or_Metastatic='Primary',
                                 further_pre=True)

../../Data/OVC/2100-Ovariancancer/matrix.mtx
../../Data/OVC/2100-Ovariancancer/barcodes.tsv
../../Data/OVC/2100-Ovariancancer/genes.tsv
../../Data/OVC/2100-Ovariancancer/2101-Ovariancancer_metadata.csv
Loading: ../../Data/OVC/2100-Ovariancancer/matrix.mtx


,0
0,BT1303_AAACCTGAGAGCCCAA
1,BT1303_AAACCTGAGGCATGGT
2,BT1303_AAACCTGAGTATTGGA
3,BT1303_AAACCTGAGTGCAAGC
4,BT1303_AAACCTGAGTGGTAAT
...,...
45109,scrSOL007_TTTGTCAGTACCGAGA
45110,scrSOL007_TTTGTCAGTGAAAGAG
45111,scrSOL007_TTTGTCAGTGAAGGCT
45112,scrSOL007_TTTGTCATCAGCCTAA


,0,1
0,RP11-34P13.3,RP11-34P13.3
1,FAM138A,FAM138A
2,OR4F5,OR4F5
3,RP11-34P13.7,RP11-34P13.7
4,RP11-34P13.8,RP11-34P13.8
...,...,...
33689,AC233755.2,AC233755.2
33690,AC233755.1,AC233755.1
33691,AC240274.1,AC240274.1
33692,AC213203.1,AC213203.1


,RP11-34P13.3,FAM138A,OR4F5,RP11-34P13.7,RP11-34P13.8,RP11-34P13.14,RP11-34P13.9,FO538757.3,FO538757.2,AP006222.2,...,AC007325.2,BX072566.1,AL354822.1,AC023491.2,AC004556.1,AC233755.2,AC233755.1,AC240274.1,AC213203.1,FAM231B
BT1303_AAACCTGAGAGCCCAA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1303_AAACCTGAGGCATGGT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1303_AAACCTGAGTATTGGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1303_AAACCTGAGTGCAAGC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1303_AAACCTGAGTGGTAAT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scrSOL007_TTTGTCAGTACCGAGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrSOL007_TTTGTCAGTGAAAGAG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrSOL007_TTTGTCAGTGAAGGCT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrSOL007_TTTGTCATCAGCCTAA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Running doublet detection on 45114 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.82
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.1%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 3.5%
  Detected 1 doublets (0.0%)
  After doublet removal: 45113 cells


/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1086: NumbaDeprecationWarning:

Finished processing 2100-Ovariancancer. Shape: (43342, 33694)


In [19]:
ad.obs['PatientNumber'] = ad.obs['PatientNumber'].astype(str)

In [14]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='CellType', 
                          celltypes_to_keep=['Cancer'],
                          further_pre=True)
ad

Original shape: (43342, 33694)
Filtered shape: (13161, 33694)
Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 13161 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'CellType_colors', 'PatientNumber_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [24]:
patient_cell_number = pd.read_csv("../../Data/OVC/2100-Ovariancancer/2101-Ovariancancer_metadata.csv")['PatientNumber'].value_counts()
patient_cell_number = patient_cell_number.to_dict()
patient_cell_number

{11: 22255, 14: 6257, 15: 6001, 13: 5450, 12: 5151}

In [25]:
pd.read_csv('../../Data/BRCA/2102-Breastcancer/Sequencing_quality_S2.txt', sep='\t')['Cancer type'].value_counts()

Cancer type
LC     36
CRC    21
BC     19
OvC    10
Name: count, dtype: int64

In [26]:
patient_seuqncing_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Sequencing_quality_S2.txt', sep='\t')
patient_seuqncing_meta_df = patient_seuqncing_meta_df[patient_seuqncing_meta_df['Cancer type'] == 'OvC']
patient_seuqncing_meta_df

,Patient number,Cancer type,10X version,Cells,Sample type,Tumour site,UMIs,Saturation (%),Reads
36,OvC_1,OvC,3' V2,8623,Normal,Omentum,28052375,73.5,350096261
37,OvC_1,OvC,3' V2,4348,Normal,Omentum,18068251,82.5,339448488
38,OvC_1,OvC,3' V2,1432,Tumour,Peritoneum,4355115,93.9,250974825
39,OvC_1,OvC,3' V2,1501,Tumour,Peritoneum,4958614,92.9,280613748
40,OvC_1,OvC,3' V2,6351,Tumour,Ovarium,31083122,78.2,400459655
41,OvC_2,OvC,3' V2,5151,Tumour,Peritoneum,25746205,39.7,193126055
42,OvC_3,OvC,3' V2,5450,Tumour,Peritoneum,26667830,81.1,306251034
43,OvC_4,OvC,3' V2,6257,Tumour,Peritoneum,44446808,63.0,298554801
44,OvC_5,OvC,3' V2,1136,Tumour,Ovarium,5213201,87.5,146559477
45,OvC_5,OvC,3' V2,4865,Normal,Ovarium,19384194,67.5,120336815


In [27]:
# Group by 'Patient number' and sum the 'Cells' column
cells_to_lc_label = patient_seuqncing_meta_df.groupby("Patient number")["Cells"].sum().to_dict()

# Flip the dict so it's {cell_sum: patient_id}
cells_to_lc_label = {v: k for k, v in cells_to_lc_label.items()}
cells_to_lc_label


{22255: 'OvC_1', 5151: 'OvC_2', 5450: 'OvC_3', 6257: 'OvC_4', 6001: 'OvC_5'}

In [28]:
patient_number_to_LC_id = dict()
for patient_number in patient_cell_number.keys():
    # print(patient_number)
    try:
        patient_number_to_LC_id[str(patient_number)] = cells_to_lc_label[patient_cell_number[patient_number]]
    except:
        print(patient_number)
patient_number_to_LC_id

{'11': 'OvC_1', '14': 'OvC_4', '15': 'OvC_5', '13': 'OvC_3', '12': 'OvC_2'}

In [29]:
ad.obs['BC_PatientID'] = ad.obs['PatientNumber'].map(patient_number_to_LC_id)
ad.obs

,Cell,nGene,nUMI,CellFromTumor,PatientNumber,TumorType,TumorSite,CellType,doublet_score,predicted_doublet,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,Project_ID,Primary_or_Metastatic,BC_PatientID
BT1303_AAACCTGAGAGCCCAA,BT1303_AAACCTGAGAGCCCAA,208,951,1,11,Ovarian,Omentum,Cancer,0.016895,False,208,951.0,7.0,0.736067,2100-Ovariancancer,Primary,OvC_1
BT1303_AAACCTGAGTATTGGA,BT1303_AAACCTGAGTATTGGA,226,407,1,11,Ovarian,Omentum,Cancer,0.130779,False,226,407.0,79.0,19.410320,2100-Ovariancancer,Primary,OvC_1
BT1303_AAACCTGAGTGGTAAT,BT1303_AAACCTGAGTGGTAAT,259,409,1,11,Ovarian,Omentum,Cancer,0.037089,False,259,409.0,7.0,1.711491,2100-Ovariancancer,Primary,OvC_1
BT1303_AAACCTGCAAGCGCTC,BT1303_AAACCTGCAAGCGCTC,303,452,1,11,Ovarian,Omentum,Cancer,0.027389,False,303,452.0,24.0,5.309734,2100-Ovariancancer,Primary,OvC_1
BT1303_AAACCTGGTAAATGTG,BT1303_AAACCTGGTAAATGTG,268,402,1,11,Ovarian,Omentum,Cancer,0.032072,False,268,402.0,19.0,4.726368,2100-Ovariancancer,Primary,OvC_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scrSOL004_TTTGTCATCGAACTGT,scrSOL004_TTTGTCATCGAACTGT,439,617,1,14,Ovarian,Peritoneum,Cancer,0.029823,False,439,617.0,47.0,7.617504,2100-Ovariancancer,Primary,OvC_4
scrSOL006_CACATAGTCCGAGCCA,scrSOL006_CACATAGTCCGAGCCA,523,925,1,15,Ovarian,Ovarium,Cancer,0.055954,False,523,925.0,15.0,1.621622,2100-Ovariancancer,Primary,OvC_5
scrSOL006_CATCGGGCATAACCTG,scrSOL006_CATCGGGCATAACCTG,760,1283,1,15,Ovarian,Ovarium,Cancer,0.087379,False,760,1283.0,23.0,1.792673,2100-Ovariancancer,Primary,OvC_5
scrSOL007_GAATGAAAGTGAACAT,scrSOL007_GAATGAAAGTGAACAT,2673,6389,0,15,Ovarian,Ovarium,Cancer,0.056686,False,2673,6389.0,532.0,8.326812,2100-Ovariancancer,Primary,OvC_5


In [30]:
patient_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Patient_metadata_S1.txt', sep='\t')
patient_meta_df = patient_meta_df[patient_meta_df['Tumor_type'] == 'OvC']
meta_subset = patient_meta_df
meta_subset

,Patient_number,Tumor_type,Gender,Age_range,Stage,TNM,Pathological_subtype,Molecular_status
8,OvC_1,OvC,Female,70-75,IIIC,pT3cNxM0,High grade serous carcinoma,NaN
9,OvC_2,OvC,Female,50-55,IVB,pT3cNxM1b,High grade serous carcinoma,NaN
10,OvC_3,OvC,Female,60-65,IVB,pT3cNxM1b,High grade serous carcinoma,BRCA+
11,OvC_4,OvC,Female,80-85,IVB,pT3cNxM1b,High grade serous + clear cell carcinoma (mix),NaN
12,OvC_5,OvC,Female,60-65,IA,pT1cN0M0,High grade serous carcinoma,NaN


In [31]:
ad.obs = ad.obs.merge(meta_subset, left_on='BC_PatientID', right_on='Patient_number', how='left')
ad.obs

,Cell,nGene,nUMI,CellFromTumor,PatientNumber,TumorType,TumorSite,CellType,doublet_score,predicted_doublet,...,Primary_or_Metastatic,BC_PatientID,Patient_number,Tumor_type,Gender,Age_range,Stage,TNM,Pathological_subtype,Molecular_status
0,BT1303_AAACCTGAGAGCCCAA,208,951,1,11,Ovarian,Omentum,Cancer,0.016895,False,...,Primary,OvC_1,OvC_1,OvC,Female,70-75,IIIC,pT3cNxM0,High grade serous carcinoma,NaN
1,BT1303_AAACCTGAGTATTGGA,226,407,1,11,Ovarian,Omentum,Cancer,0.130779,False,...,Primary,OvC_1,OvC_1,OvC,Female,70-75,IIIC,pT3cNxM0,High grade serous carcinoma,NaN
2,BT1303_AAACCTGAGTGGTAAT,259,409,1,11,Ovarian,Omentum,Cancer,0.037089,False,...,Primary,OvC_1,OvC_1,OvC,Female,70-75,IIIC,pT3cNxM0,High grade serous carcinoma,NaN
3,BT1303_AAACCTGCAAGCGCTC,303,452,1,11,Ovarian,Omentum,Cancer,0.027389,False,...,Primary,OvC_1,OvC_1,OvC,Female,70-75,IIIC,pT3cNxM0,High grade serous carcinoma,NaN
4,BT1303_AAACCTGGTAAATGTG,268,402,1,11,Ovarian,Omentum,Cancer,0.032072,False,...,Primary,OvC_1,OvC_1,OvC,Female,70-75,IIIC,pT3cNxM0,High grade serous carcinoma,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13156,scrSOL004_TTTGTCATCGAACTGT,439,617,1,14,Ovarian,Peritoneum,Cancer,0.029823,False,...,Primary,OvC_4,OvC_4,OvC,Female,80-85,IVB,pT3cNxM1b,High grade serous + clear cell carcinoma (mix),NaN
13157,scrSOL006_CACATAGTCCGAGCCA,523,925,1,15,Ovarian,Ovarium,Cancer,0.055954,False,...,Primary,OvC_5,OvC_5,OvC,Female,60-65,IA,pT1cN0M0,High grade serous carcinoma,NaN
13158,scrSOL006_CATCGGGCATAACCTG,760,1283,1,15,Ovarian,Ovarium,Cancer,0.087379,False,...,Primary,OvC_5,OvC_5,OvC,Female,60-65,IA,pT1cN0M0,High grade serous carcinoma,NaN
13159,scrSOL007_GAATGAAAGTGAACAT,2673,6389,0,15,Ovarian,Ovarium,Cancer,0.056686,False,...,Primary,OvC_5,OvC_5,OvC,Female,60-65,IA,pT1cN0M0,High grade serous carcinoma,NaN


In [32]:
ad.obs['Final_cancer_type'] = 'Ovarian Cancer'
ad.obs['Final_histological_subtype'] = ad.obs.Pathological_subtype
ad.obs['Final_molecular_subtype'] = ad.obs['Molecular_status']
ad.obs['Final_tissue'] = 'Ovary'
ad.obs['Final_sample_id'] = ad.obs['BC_PatientID']

In [33]:
# Ensure TNM is string
ad.obs['TNM'] = ad.obs['TNM'].astype(str)

# If Primary_or_Metastatic is categorical, add new category first
if pd.api.types.is_categorical_dtype(ad.obs['Primary_or_Metastatic']):
    ad.obs['Primary_or_Metastatic'] = ad.obs['Primary_or_Metastatic'].cat.add_categories(['Metastatic'])

# Now assign "Metastatic" to rows where TNM contains "M1"
ad.obs.loc[ad.obs['TNM'].str.contains('M1', na=False), 'Primary_or_Metastatic'] = 'Metastatic'


In [34]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs['Age_range']
ad.obs['Final_patient_stage'] = ad.obs['TNM']
ad.obs['Final_patient_treatment'] = 'Naïve'

In [35]:
ad.raw.shape

(13161, 33694)

In [36]:
ad

AnnData object with n_obs × n_vars = 13161 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'BC_PatientID', 'Patient_number', 'Tumor_type', 'Gender', 'Age_range', 'Stage', 'TNM', 'Pathological_subtype', 'Molecular_status', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue', 'Final_sample_id', 'Final_patient_age', 'Final_patient_stage', 'Final_patient_treatment'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'CellType_colors', 'PatientNumber_colors', 'TumorType_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [37]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/2100-Ovariancancer.OVC.h5ad', compression='gzip')

## Ovarian cancer mutational processes drive site-specific immune evasion

Paper: https://www.nature.com/articles/s41586-022-05496-1

Data downloaded from: https://cellxgene.cziscience.com/collections/4796c91c-9d8f-4692-be43-347b1727f9d8

Link: 
- H5ad: https://datasets.cellxgene.cziscience.com/bc454432-4453-4eee-afa5-d5d5a0760a0d.h5ad
- More clinical metadata: https://static-content.springer.com/esm/art%3A10.1038%2Fs41586-022-05496-1/MediaObjects/41586_2022_5496_MOESM3_ESM.xlsx

In [11]:
ad = sc.read_h5ad('../../Data/OVC/MKS_SPECTRUM/MSK_SPECTRUM.h5ad')
ad

AnnData object with n_obs × n_vars = 927205 × 31815
    obs: 'percent.mt', 'percent.rb', 'doublet', 'author_sample_id', 'S.Score', 'G2M.Score', 'Phase', 'CC.Diff', 'author_cell_type', 'nCount_RNA', 'nFeature_RNA', 'doublet_score', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'author_tumor_supersite', 'author_tumor_site', 'author_tumor_subsite', 'author_sort_parameters', 'author_therapy', 'author_procedure', 'author_procedure_type', 'is_primary_data', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'author_cell_type_colors', 'citation', 'default_embedding', 'neighbors', 'orga

In [12]:
celltypes_to_keep = []
for types in ad.obs['author_cell_type'].value_counts().index:
    if types.lower().__contains__('cancer'):
        celltypes_to_keep.append(types)
celltypes_to_keep

['Ovarian.cancer.cell']

In [13]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='author_cell_type', 
                          celltypes_to_keep=celltypes_to_keep,
                          further_pre=False)
ad

Original shape: (927205, 31815)
Filtered shape: (250281, 31815)


AnnData object with n_obs × n_vars = 250281 × 31815
    obs: 'percent.mt', 'percent.rb', 'doublet', 'author_sample_id', 'S.Score', 'G2M.Score', 'Phase', 'CC.Diff', 'author_cell_type', 'nCount_RNA', 'nFeature_RNA', 'doublet_score', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'author_tumor_supersite', 'author_tumor_site', 'author_tumor_subsite', 'author_sort_parameters', 'author_therapy', 'author_procedure', 'author_procedure_type', 'is_primary_data', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'author_cell_type_colors', 'citation', 'default_embedding', 'neighbors', 'orga

In [14]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='MSK_SPECTRUM', 
                              Primary_or_Metastatic='Metastatic',
                              further_pre=False)

Running doublet detection on 250281 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.82
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 2.3%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.0%
  Detected 2 doublets (0.0%)
  After doublet removal: 250279 cells
Standard filtering...
Reprocessed dataset. Final shape: (231924, 31815)


In [16]:
ad.obs['Final_cancer_type'] = 'Ovarian Cancer'
ad.obs['Final_histological_subtype'] = 'High-Grade Serous Ovarian Cancer'
ad.obs['Final_molecular_subtype'] = 'OVC: Unspecified'
ad.obs['Final_tissue'] = ad.obs['author_tumor_supersite']
ad.obs['Final_sample_id'] = ad.obs['donor_id']

In [17]:
patient_clinical_df = pd.read_csv('../../Data/OVC/MKS_SPECTRUM/Patient_clinical.txt', sep='\t')
patient_clinical_df

,patient_id,patient_isabl_id,patient_dmp_id,patient_age_at_diagnosis,gyn_diagnosis_histology,gyn_diagnosis_chemo_intent_description,gyn_diagnosis_figo_stage
0,SPECTRUM-OV-002,SHAH_H000004,P-0039615,67,HGS,Primary,IVB
1,SPECTRUM-OV-003,SHAH_H000005,P-0039726,61,HGS,NACT/IDS,IIIC
2,SPECTRUM-OV-004,SHAH_H000029,P-0039734,71,HGS,Primary,IIIC
3,SPECTRUM-OV-007,SHAH_H000006,P-0040980,43,HGS,Primary,IIIC
4,SPECTRUM-OV-008,SHAH_H000011,P-0041572,61,HGS,Primary,IIIC
5,SPECTRUM-OV-009,SHAH_H000013,P-0042164,47,HGS,Primary,IVB
6,SPECTRUM-OV-014,SHAH_H000009,P-0042548,81,HGS,NACT/IDS,IIIC
7,SPECTRUM-OV-022,SHAH_H000019,P-0043571,55,HGS,Primary,IIIB
8,SPECTRUM-OV-024,SHAH_H000020,P-0044784,81,HGS,NACT/IDS,IIIC
9,SPECTRUM-OV-025,SHAH_H000021,P-0044084,72,HGS,Primary,IIIC


In [20]:
# Step 1: Reset index for merging, but save cell IDs
# ad.obs['tmp_donor_id'] = [i.replace('SMP-', '').replace('-', '_').replace('HTAPP', 'HTA1') for i in ad.obs['sampleid']]
merged = ad.obs.reset_index().merge(
    patient_clinical_df,
    left_on='donor_id',
    right_on='patient_id',
    how='left'
)

# Step 2: Restore original index (cell barcodes)
merged = merged.set_index('cell_id')

# Step 3: Assign back
ad.obs = merged
ad

/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:788: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAACGAAGTTTCCATT', 'SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAACGCTCACAACGAG', 'SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAAGAACAGAGCATAT', 'SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAAGGATAGACTGGGT', 'SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAAGGATGTTCACCGG']

    Inferred to be: categorical

  value_idx = self._prep_dim_index(value.index, attr)


AnnData object with n_obs × n_vars = 231924 × 31815
    obs: 'percent.mt', 'percent.rb', 'doublet', 'author_sample_id', 'S.Score', 'G2M.Score', 'Phase', 'CC.Diff', 'author_cell_type', 'nCount_RNA', 'nFeature_RNA', 'doublet_score', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'author_tumor_supersite', 'author_tumor_site', 'author_tumor_subsite', 'author_sort_parameters', 'author_therapy', 'author_procedure', 'author_procedure_type', 'is_primary_data', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtyp

In [21]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs['patient_age_at_diagnosis']
ad.obs['Final_patient_stage'] = ad.obs['gyn_diagnosis_figo_stage']
ad.obs['Final_patient_treatment'] = ad.obs['gyn_diagnosis_chemo_intent_description']

In [22]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/MKS_SPECTRUM.OVC.h5ad', compression='gzip')

## A multi-omic single-cell landscape of human gynecologic malignancies

Paper: https://www.sciencedirect.com/science/article/pii/S109727652100842X#app2

Data downloaded from: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE173682

Link: 
- Matrix: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE173682
- Cell Annotation: https://ars.els-cdn.com/content/image/1-s2.0-S109727652100842X-mmc3.xlsx
- Patient metadata: https://ars.els-cdn.com/content/image/1-s2.0-S109727652100842X-mmc2.xlsx


In [11]:
project_dir = '../../Data/OVC/GSE173682/GSE173682_raw/'
all_files = os.listdir(project_dir)
all_files.sort()
# all_files

In [12]:
all_samples = set([i.split('_')[0] for i in all_files])
all_samples = list(all_samples)
all_samples.sort()
all_samples

['GSM5276933',
 'GSM5276934',
 'GSM5276935',
 'GSM5276936',
 'GSM5276937',
 'GSM5276938',
 'GSM5276939',
 'GSM5276940',
 'GSM5276941',
 'GSM5276942',
 'GSM5276943']

In [13]:
ad_lsit = []
for sample in all_samples:
    print(sample)
    ad = load_and_preprocess_project(project_dir, 
                                     Project_ID='GSE173682', 
                                     Primary_or_Metastatic='Primary',
                                     further_pre=False,
                                     file_prefix=sample)
    print(ad)
    ad_lsit.append(ad)

GSM5276933
../../Data/OVC/GSE173682/GSE173682_raw/GSM5276933_features-3533EL.tsv
../../Data/OVC/GSE173682/GSE173682_raw/GSM5276933_barcodes-3533EL.tsv
../../Data/OVC/GSE173682/GSE173682_raw/GSM5276933_matrix-3533EL.mtx
Loading: ../../Data/OVC/GSE173682/GSE173682_raw/GSM5276933_matrix-3533EL.mtx


,0
0,AAACCCACACGCCAGT-1
1,AAACCCACATTGACAC-1
2,AAACCCAGTGACTAAA-1
3,AAACCCAGTTGCATAC-1
4,AAACCCATCGCCGAGT-1
...,...
5692,TTTGTTGGTGTTGAGG-1
5693,TTTGTTGGTTGCAACT-1
5694,TTTGTTGGTTGTGTTG-1
5695,TTTGTTGTCAAGCTTG-1


,0,1,2
0,ENSG00000243485,MIR1302-2HG,Gene Expression
1,ENSG00000237613,FAM138A,Gene Expression
2,ENSG00000186092,OR4F5,Gene Expression
3,ENSG00000238009,AL627309.1,Gene Expression
4,ENSG00000239945,AL627309.3,Gene Expression
...,...,...,...
33533,ENSG00000277856,AC233755.2,Gene Expression
33534,ENSG00000275063,AC233755.1,Gene Expression
33535,ENSG00000271254,AC240274.1,Gene Expression
33536,ENSG00000277475,AC213203.1,Gene Expression


,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.4,AL732372.1,OR4F29,AC114498.1,...,AC007325.2,BX072566.1,AL354822.1,AC023491.2,AC004556.1,AC233755.2,AC233755.1,AC240274.1,AC213203.1,FAM231C
AAACCCACACGCCAGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCCACATTGACAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
AAACCCAGTGACTAAA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCCAGTTGCATAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCCATCGCCGAGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTGTTGAGG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGGTTGCAACT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGGTTGTGTTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGTCAAGCTTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Running doublet detection on 5697 cells...


...

/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making

Automatically set threshold at doublet score = 0.29
Detected doublet rate = 1.6%
Estimated detectable doublet fraction = 40.3%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 3.9%
  Detected 109 doublets (1.6%)
  After doublet removal: 6830 cells


/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Finished processing GSE173682. Shape: (5484, 33538)
AnnData object with n_obs × n_vars = 5484 × 33538
    obs: 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'scrublet'


/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [16]:
for ad in ad_lsit:
    ad.var_names_make_unique()
    ad.raw = ad

for i, ad in enumerate(ad_lsit):
    if ad.raw is not None:
        if not ad.raw.var_names.is_unique:
            print(f"AnnData {i} has duplicate raw.var_names!")


In [17]:
for i, ad in enumerate(ad_lsit):
    if ad.raw is not None:
        raw_var_cols = ad.raw.var.columns
        if not raw_var_cols.is_unique:
            print(f"AnnData {i} has duplicated columns in .raw.var: {raw_var_cols[raw_var_cols.duplicated()].tolist()}")


In [18]:
combined_ad = ann.concat(ad_lsit, join="inner", axis=0)
combined_ad

/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 72028 × 33538
    obs: 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'

In [19]:
metadata_df = pd.read_csv('../../Data/OVC/GSE173682/meta_all.txt', sep='\t', index_col=0)
metadata_df

,nCount_RNA,nFeature_RNA,RNA_snn_res.0.7,seurat_clusters,PC1.loading,CNV.Pos,Total_CNVs,SingleR,Sample,percent.mt,cell.type
Barcode,,,,,,,,,,,
AAACCCACACGCCAGT-1_1,1655,868,4,4,5.180581,False,0,Lymphocytes,3533EL,10.513595,4-Lymphocytes
AAACCCACATTGACAC-1_1,21094,4275,31,31,-5.329042,True,3,Unciliated epithelia 1,3533EL,4.840239,31-Unciliated epithelia 1
AAACCCAGTGACTAAA-1_1,5316,1542,35,35,5.029341,False,0,Macrophages,3533EL,13.167795,35-B cell
AAACCCAGTTGCATAC-1_1,2146,852,7,7,4.079348,False,0,Smooth muscle cells,3533EL,17.381174,7-Smooth muscle cells
AAACCCATCGCCGAGT-1_1,1863,764,7,7,4.565346,False,0,Stromal fibroblasts,3533EL,8.105207,7-Smooth muscle cells
...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTCCTTGTC-1_11,16624,4065,9,9,-6.958114,True,5,Epithelial cell,3E5CFL,3.152069,9-Epithelial cell
TTTGTTGGTCTTTCAT-1_11,10377,3092,18,18,4.887620,False,0,Fibroblast,3E5CFL,1.936976,18-Fibroblast
TTTGTTGGTTCCGTTC-1_11,2212,1113,9,9,-3.106266,True,5,Empty/Epithelial cell,3E5CFL,0.271248,9-Epithelial cell


In [20]:
metadata_df = metadata_df[~metadata_df.index.duplicated(keep='first')]
metadata_df

,nCount_RNA,nFeature_RNA,RNA_snn_res.0.7,seurat_clusters,PC1.loading,CNV.Pos,Total_CNVs,SingleR,Sample,percent.mt,cell.type
Barcode,,,,,,,,,,,
AAACCCACACGCCAGT-1_1,1655,868,4,4,5.180581,False,0,Lymphocytes,3533EL,10.513595,4-Lymphocytes
AAACCCACATTGACAC-1_1,21094,4275,31,31,-5.329042,True,3,Unciliated epithelia 1,3533EL,4.840239,31-Unciliated epithelia 1
AAACCCAGTGACTAAA-1_1,5316,1542,35,35,5.029341,False,0,Macrophages,3533EL,13.167795,35-B cell
AAACCCAGTTGCATAC-1_1,2146,852,7,7,4.079348,False,0,Smooth muscle cells,3533EL,17.381174,7-Smooth muscle cells
AAACCCATCGCCGAGT-1_1,1863,764,7,7,4.565346,False,0,Stromal fibroblasts,3533EL,8.105207,7-Smooth muscle cells
...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTCCTTGTC-1_11,16624,4065,9,9,-6.958114,True,5,Epithelial cell,3E5CFL,3.152069,9-Epithelial cell
TTTGTTGGTCTTTCAT-1_11,10377,3092,18,18,4.887620,False,0,Fibroblast,3E5CFL,1.936976,18-Fibroblast
TTTGTTGGTTCCGTTC-1_11,2212,1113,9,9,-3.106266,True,5,Empty/Epithelial cell,3E5CFL,0.271248,9-Epithelial cell


In [21]:
combined_ad.obs_names_make_unique()
combined_ad

AnnData object with n_obs × n_vars = 72028 × 33538
    obs: 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'

In [22]:
shared_cells = set([i.split('_')[0] for i in metadata_df.index]).intersection(set([i.split('_')[0] for i in combined_ad.to_df().index]))
len(shared_cells)

66258

In [23]:
new_index = [i.split('_')[0] for i in metadata_df.index]
metadata_df.index = new_index

In [24]:
shared_cells = list(shared_cells)
shared_cells.sort()
metadata_df = metadata_df[~metadata_df.index.duplicated(keep='first')]
metadata_df
combined_ad = combined_ad[shared_cells]
combined_ad.obs = metadata_df.loc[combined_ad.obs.index]

In [25]:
ovc_sample = set(['38FE7L', '3BAE2L', '3CCF1L', '3E4D1L', '3E5CFL'])

In [26]:
combined_ad.obs

,nCount_RNA,nFeature_RNA,RNA_snn_res.0.7,seurat_clusters,PC1.loading,CNV.Pos,Total_CNVs,SingleR,Sample,percent.mt,cell.type
AAACCCAAGACCAAAT-1,8417,2560,8,8,4.068286,True,1,Stromal fibroblasts,3571DL,3.088987,8-Stromal fibroblasts
AAACCCAAGAGAGAAC-1,18998,4797,11,11,-7.311395,False,0,Unciliated epithelia 1,36186L,6.353300,11-Unciliated epithelia 1
AAACCCAAGATCCAAA-1,9176,2806,3,3,-8.731941,False,0,Epithelial cell,38FE7L,19.049695,3-Epithelial cell
AAACCCAAGCATAGGC-1,1956,919,4,4,4.312078,False,0,Lymphocytes,36639L,9.304703,4-Lymphocytes
AAACCCAAGCATTTCG-1,10887,2480,22,22,-5.770916,False,0,Unciliated epithelia 1,366C5L,1.405346,22-Unciliated epithelia 2
...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCTAGATCG-1,3899,1509,14,14,3.748360,False,0,Stromal fibroblasts,366C5L,10.207746,14-Stromal fibroblasts
TTTGTTGTCTCTGGTC-1,10338,3787,17,17,-5.364955,True,6,Empty/Epithelial cell,3CCF1L,1.209131,17-Epithelial cell
TTTGTTGTCTGTGTGA-1,2623,878,17,17,0.724663,False,0,NaN,366C5L,0.114373,17-Epithelial cell
TTTGTTGTCTTCTTCC-1,4488,1938,0,0,3.371062,False,0,Fibroblast,3E4D1L,1.158645,0-Fibroblast


In [27]:
combined_ad = combined_ad[combined_ad.obs['CNV.Pos'] == True]
combined_ad

View of AnnData object with n_obs × n_vars = 16262 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.7', 'seurat_clusters', 'PC1.loading', 'CNV.Pos', 'Total_CNVs', 'SingleR', 'Sample', 'percent.mt', 'cell.type'

In [28]:
combined_ad = filter_and_recompute(adata=combined_ad, 
                          celltype_col='Sample', 
                          celltypes_to_keep=ovc_sample,
                          further_pre=False)
combined_ad

Original shape: (16262, 33538)
Filtered shape: (10643, 33538)


AnnData object with n_obs × n_vars = 10643 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.7', 'seurat_clusters', 'PC1.loading', 'CNV.Pos', 'Total_CNVs', 'SingleR', 'Sample', 'percent.mt', 'cell.type'

In [29]:
combined_ad.obs['cell.type'].value_counts()

cell.type
0-Fibroblast                 3839
9-Epithelial cell            1898
16-Fibroblast                1882
17-Epithelial cell           1396
10-Epithelial cell            949
27-Fibroblast                 455
30-T cell                     148
3-Epithelial cell              50
7-Smooth muscle cells           8
28-B cell                       6
11-Unciliated epithelia 1       4
20-Ciliated                     2
14-Stromal fibroblasts          2
35-B cell                       1
1-Endothelia                    1
5-Macrophage                    1
23-Stromal fibroblasts          1
Name: count, dtype: int64

In [30]:
kept_cell_types = []
for cell_type in combined_ad.obs['cell.type'].value_counts().index:
    if cell_type.__contains__('Epithelial'):
        kept_cell_types.append(cell_type)
kept_cell_types

['9-Epithelial cell',
 '17-Epithelial cell',
 '10-Epithelial cell',
 '3-Epithelial cell']

In [31]:
combined_ad = filter_and_recompute(adata=combined_ad, 
                          celltype_col='cell.type', 
                          celltypes_to_keep=kept_cell_types,
                          further_pre=True)
combined_ad

Original shape: (10643, 33538)
Filtered shape: (4293, 33538)


/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1086: NumbaDeprecationWarning:

Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 4293 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.7', 'seurat_clusters', 'PC1.loading', 'CNV.Pos', 'Total_CNVs', 'SingleR', 'Sample', 'percent.mt', 'cell.type'
    uns: 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [ ]:
reset_plot()
sc.pl.umap(combined_ad, color=['Sample', 'cell.type', 'Total_CNVs'])

In [33]:
patient_clinical_df = pd.read_csv('../../Data/OVC/GSE173682/patient_clinical.txt', sep='\t')
patient_clinical_df

,Type,Tumor ID,Tumor Histology,Cell Viability,scRNA Lib Conc. (1:10 dilution),scRNA Index,scATAC Lib Conc. (1:10 dilution),scATAC Index,Patient name,Lab ID,Collection Year,Age at dx,Race,BMI,Tumor histology,Tumor grade,Stage
0,Endometrial,3533EL,Endometrioid,68%,unkwn,F6,1160 pg/ul,A3,Patient 1,3533EL,2019,70,AA,39.89,Endometrioid,2,IA
1,Endometrial,3571DL,Endometrioid,65%,unkwn,G1,unkwn,A4,Patient 2,3571DL,2019,70,CAU,30.50,Endometrioid,3,IA
2,Endometrial,36186L,Endometrioid,68%,unkwn,G7,5160 pg/ul,A7,Patient 3,36186L,2019,70,CAU,38.55,Endometrioid,2,IA
3,Endometrial,36639L,Endometrioid,74%,unkwn,G10,6410 pg/ul,A8,Patient 4,36639L,2019,49,CAU,55.29,Endometrioid,1,IA
4,Endometrial,366C5L,Endometrioid,74%,unkwn,G11,5780 pg/ul,A9,Patient 5,366C5L,2019,62,CAU,49.44,Endometrioid,1,IA
5,Endometrial (metastasis to ovary),37EACL,Serous,66%,2900 pg/ul,H2,2760 pg/ul,A10,Patient 6,37EACL,2019,74,CAU,29.94,Serous,3,IIIA
6,Ovarian,38FE7L,Endometrioid,64%,unkwn,H8,unkwn,A12,Patient 7,38FE7L,2019,76,CAU,34.80,Endometrioid,1,IA
7,Ovarian,3BAE2L,High grade serous,73%,1910 pg/ul,A9,2480 pg/ul,B2,Patient 8,3BAE2L,2019,61,CAU,22.13,High grade serous,.,IIB
8,Ovarian,3CCF1L,Carcinosarcoma,94%,1470 pg/ul,A11,6430 pg/ul,B4,Patient 10,3CCF1L,2020,69,CAU,23.72,Carcinosarcoma,.,IVB
9,Ovarian,3E4D1L,GIST,95%,1240 pg/ul,B3,4180 pg/ul,B6,Patient 11,3E4D1L,2020,59,CAU,33.96,GIST,.,IV


In [34]:
combined_ad.obs

,nCount_RNA,nFeature_RNA,RNA_snn_res.0.7,seurat_clusters,PC1.loading,CNV.Pos,Total_CNVs,SingleR,Sample,percent.mt,cell.type
AAACCCACACTGGCGT-1,9797,3129,9,9,-3.761060,True,8,Epithelial cell,3E5CFL,7.747270,9-Epithelial cell
AAACCCACAGGTACGA-1,4088,1433,9,9,-5.639114,True,2,Empty/Epithelial cell,3E5CFL,0.342466,9-Epithelial cell
AAACCCACATACAGCT-1,4697,1630,3,3,-5.931484,True,1,Epithelial cell,38FE7L,16.989568,3-Epithelial cell
AAACGAAAGTACCATC-1,10850,2973,9,9,-8.743764,True,5,Epithelial cell,3E5CFL,4.460829,9-Epithelial cell
AAACGAAAGTGTGTTC-1,26419,4360,9,9,-9.292474,True,5,Epithelial cell,3E5CFL,7.763352,9-Epithelial cell
...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGAGTGAGGCT-1,7863,2564,10,10,-6.980577,True,4,Epithelial cell,3BAE2L,13.023019,10-Epithelial cell
TTTGTTGGTATGCGTT-1,11133,3024,10,10,-10.100392,True,4,Epithelial cell,3BAE2L,12.674032,10-Epithelial cell
TTTGTTGGTTCCGTTC-1,2212,1113,9,9,-3.106266,True,5,Empty/Epithelial cell,3E5CFL,0.271248,9-Epithelial cell
TTTGTTGTCCGTCAAA-1,13284,3209,9,9,-5.624542,True,8,Epithelial cell,3E5CFL,9.552846,9-Epithelial cell


In [35]:
# Step 1: Reset index for merging, but save cell IDs
merged = combined_ad.obs.reset_index().merge(
    patient_clinical_df,
    left_on='Sample',
    right_on='Tumor ID',
    how='left'
)

# Step 2: Restore original index (cell barcodes)
merged = merged.set_index('index')

# Step 3: Assign back
combined_ad.obs = merged
combined_ad.obs

,nCount_RNA,nFeature_RNA,RNA_snn_res.0.7,seurat_clusters,PC1.loading,CNV.Pos,Total_CNVs,SingleR,Sample,percent.mt,...,scATAC Index,Patient name,Lab ID,Collection Year,Age at dx,Race,BMI,Tumor histology,Tumor grade,Stage
index,,,,,,,,,,,,,,,,,,,,,
AAACCCACACTGGCGT-1,9797,3129,9,9,-3.761060,True,8,Epithelial cell,3E5CFL,7.747270,...,B7,Patient 9,3E5CFL,2020,59,Asian,22.37,High grade serous,.,IIIC
AAACCCACAGGTACGA-1,4088,1433,9,9,-5.639114,True,2,Empty/Epithelial cell,3E5CFL,0.342466,...,B7,Patient 9,3E5CFL,2020,59,Asian,22.37,High grade serous,.,IIIC
AAACCCACATACAGCT-1,4697,1630,3,3,-5.931484,True,1,Epithelial cell,38FE7L,16.989568,...,A12,Patient 7,38FE7L,2019,76,CAU,34.80,Endometrioid,1,IA
AAACGAAAGTACCATC-1,10850,2973,9,9,-8.743764,True,5,Epithelial cell,3E5CFL,4.460829,...,B7,Patient 9,3E5CFL,2020,59,Asian,22.37,High grade serous,.,IIIC
AAACGAAAGTGTGTTC-1,26419,4360,9,9,-9.292474,True,5,Epithelial cell,3E5CFL,7.763352,...,B7,Patient 9,3E5CFL,2020,59,Asian,22.37,High grade serous,.,IIIC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGAGTGAGGCT-1,7863,2564,10,10,-6.980577,True,4,Epithelial cell,3BAE2L,13.023019,...,B2,Patient 8,3BAE2L,2019,61,CAU,22.13,High grade serous,.,IIB
TTTGTTGGTATGCGTT-1,11133,3024,10,10,-10.100392,True,4,Epithelial cell,3BAE2L,12.674032,...,B2,Patient 8,3BAE2L,2019,61,CAU,22.13,High grade serous,.,IIB
TTTGTTGGTTCCGTTC-1,2212,1113,9,9,-3.106266,True,5,Empty/Epithelial cell,3E5CFL,0.271248,...,B7,Patient 9,3E5CFL,2020,59,Asian,22.37,High grade serous,.,IIIC


In [36]:
primary_or_mets = []
for stage in combined_ad.obs['Stage']:
    if stage.startswith('IV'):
        primary_or_mets.append('Metastatic')
    else:
        primary_or_mets.append('Primary')

In [37]:
# add more clinical information
combined_ad.obs['Project_ID'] = 'GSE173682'
combined_ad.obs['Primary_or_Metastatic'] = primary_or_mets
combined_ad.obs['Final_cancer_type'] = 'Ovarian Cancer'
combined_ad.obs['Final_histological_subtype'] = combined_ad.obs['Tumor histology']
combined_ad.obs['Final_molecular_subtype'] = 'Unknown'
combined_ad.obs['Final_tissue'] = 'Ovary'
combined_ad.obs['Final_sample_id'] = combined_ad.obs['Sample']
combined_ad.obs['Final_patient_age'] = combined_ad.obs['Age at dx']
combined_ad.obs['Final_patient_stage'] = combined_ad.obs['Stage']
combined_ad.obs['Final_patient_treatment'] = 'Naive'

In [38]:
combined_ad

AnnData object with n_obs × n_vars = 4293 × 33538
    obs: 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.7', 'seurat_clusters', 'PC1.loading', 'CNV.Pos', 'Total_CNVs', 'SingleR', 'Sample', 'percent.mt', 'cell.type', 'Type', 'Tumor ID', 'Tumor Histology', 'Cell Viability', 'scRNA Lib Conc. (1:10 dilution)', 'scRNA Index', 'scATAC Lib Conc. (1:10 dilution)', 'scATAC Index', 'Patient name', 'Lab ID', 'Collection Year', 'Age at dx', 'Race', 'BMI', 'Tumor histology', 'Tumor grade', 'Stage', 'Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue', 'Final_sample_id', 'Final_patient_age', 'Final_patient_stage', 'Final_patient_treatment'
    uns: 'pca', 'neighbors', 'umap', 'Sample_colors', 'cell.type_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [39]:
combined_ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/GSE173682.OVC.h5ad', compression='gzip')

## Probabilistic cell-type assignment of single-cell RNA-seq for tumor microenvironment profiling


Paper: https://www.nature.com/articles/s41592-019-0529-1

Data downloaded from: https://www.weizmann.ac.il/sites/3CA/ovarian (Zhang et al. 2019 )
Link: 
- Matrix: https://www.dropbox.com/scl/fi/xodvbb2ozhyde2rm83q9c/Data_Zhang2019_Ovarian.tar.gz?rlkey=r2p1s9uj370854h226llzxpt6&dl=1
- Metadata: https://www.dropbox.com/scl/fi/drs9docefehgjdo7mqs5q/Meta-data_Zhang2019_Ovarian.tar.gz?rlkey=7nxr2m3tis24j91upl28oybza&dl=1

In [23]:
# Set the project directory
project_dir = "../../Data/OVC/Zhang_2019/Data_Zhang2019_Ovarian/"  # <- change this for each project

# Load and preprocess
ad = load_and_preprocess_project(project_dir, 
                                 Project_ID='Zhang_2019', 
                                 Primary_or_Metastatic='Primary',
                                 further_pre=False)
ad

../../Data/OVC/Zhang_2019/Data_Zhang2019_Ovarian/Exp_data_UMIcounts.mtx
../../Data/OVC/Zhang_2019/Data_Zhang2019_Ovarian/barcodes.tsv
../../Data/OVC/Zhang_2019/Data_Zhang2019_Ovarian/Genes.txt
../../Data/OVC/Zhang_2019/Data_Zhang2019_Ovarian/meta.csv
Loading: ../../Data/OVC/Zhang_2019/Data_Zhang2019_Ovarian/Exp_data_UMIcounts.mtx


,0
0,AAACCTGAGAGACGAA_VOA11543L
1,AAACCTGAGAGTCTGG_VOA11543L
2,AAACCTGAGATCCCGC_VOA11543L
3,AAACCTGCACAGGAGT_VOA11543L
4,AAACCTGGTACTTGAC_VOA11543L
...,...
4843,TTTGGTTGTGTAACGG_VOA11543R
4844,TTTGGTTGTTCATGGT_VOA11543R
4845,TTTGGTTTCATAGCAC_VOA11543R
4846,TTTGGTTTCTTGTCAT_VOA11543R


,GNames
0,MIR1302-2HG
1,FAM138A
2,OR4F5
3,DDX11L17
4,WASH9P
...,...
24405,MT-ND4
24406,MT-ND5
24407,MT-ND6
24408,MT-CYB


,MIR1302-2HG,FAM138A,OR4F5,DDX11L17,WASH9P,LINC01409,FAM87B,LINC00115,FAM41C,LINC02593,...,MT-ATP8,MT-ATP6,MT-CO3,MT-ND3,MT-ND4L,MT-ND4,MT-ND5,MT-ND6,MT-CYB,MAFIP
AAACCTGAGAGACGAA_VOA11543L,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,102.0,113.0,103.0,104.0,61.0,46.0,25.0,3.0,61.0,0.0
AAACCTGAGAGTCTGG_VOA11543L,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,49.0,66.0,58.0,57.0,39.0,42.0,9.0,3.0,34.0,0.0
AAACCTGAGATCCCGC_VOA11543L,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,223.0,311.0,281.0,368.0,297.0,257.0,60.0,13.0,258.0,0.0
AAACCTGCACAGGAGT_VOA11543L,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,67.0,104.0,151.0,112.0,127.0,123.0,34.0,2.0,73.0,0.0
AAACCTGGTACTTGAC_VOA11543L,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,69.0,95.0,98.0,85.0,63.0,49.0,29.0,3.0,63.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTGTGTAACGG_VOA11543R,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,30.0,31.0,27.0,47.0,54.0,19.0,13.0,1.0,40.0,0.0
TTTGGTTGTTCATGGT_VOA11543R,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,123.0,109.0,172.0,218.0,150.0,79.0,38.0,3.0,176.0,0.0
TTTGGTTTCATAGCAC_VOA11543R,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,1.0,0.0,...,44.0,34.0,54.0,50.0,98.0,71.0,42.0,3.0,87.0,0.0
TTTGGTTTCTTGTCAT_VOA11543R,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,...,35.0,38.0,11.0,43.0,28.0,11.0,8.0,0.0,15.0,0.0


Running doublet detection on 4848 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.44
Detected doublet rate = 0.5%
Estimated detectable doublet fraction = 32.8%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 1.4%
  Detected 22 doublets (0.5%)
  After doublet removal: 4826 cells
Finished processing Zhang_2019. Shape: (3214, 24410)


AnnData object with n_obs × n_vars = 3214 × 24410
    obs: 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'scrublet'

In [24]:
meta_data_df = pd.read_csv('../../Data/OVC/Zhang_2019/Meta-data_Zhang2019_Ovarian/Cells.csv', index_col=0)
meta_data_df

,sample,patient,cell_type,complexity,umap1,umap2,g1s_score,g2m_score,cell_cycle_phase,mp_top_score,mp_top,mp_assignment,source
cell_name,,,,,,,,,,,,,
AAACCTGAGAGACGAA_VOA11543L,VOA11543L,VOA11543,Fibroblast,4714,12.1482,27.5697,-0.1440,-0.0600,Not cycling,0.5393,CAF10,NaN,Left ovary
AAACCTGAGAGTCTGG_VOA11543L,VOA11543L,VOA11543,Fibroblast,2103,29.2269,9.9902,0.0844,-0.0102,Not cycling,0.7347,Complement,NaN,Left ovary
AAACCTGAGATCCCGC_VOA11543L,VOA11543L,VOA11543,Malignant,5107,-23.8264,13.2392,0.0933,-0.0826,Not cycling,0.5151,Translation initiation,NaN,Left ovary
AAACCTGCACAGGAGT_VOA11543L,VOA11543L,VOA11543,Malignant,4820,-24.3399,12.6990,-0.0118,0.0163,Not cycling,0.5252,Stress,NaN,Left ovary
AAACCTGGTACTTGAC_VOA11543L,VOA11543L,VOA11543,T_cell,3452,25.6163,12.0930,NaN,NaN,NaN,NaN,NaN,NaN,Left ovary
...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTGTGTAACGG_VOA11543R,VOA11543R,VOA11543,Malignant,2946,-5.3561,-15.7878,-0.1851,-0.0177,Not cycling,1.2055,Chromatin,NaN,Right ovary
TTTGGTTGTTCATGGT_VOA11543R,VOA11543R,VOA11543,Malignant,4321,-6.0200,-10.7464,0.1735,-0.0244,Not cycling,1.2051,Stress,Stress,Right ovary
TTTGGTTTCATAGCAC_VOA11543R,VOA11543R,VOA11543,Malignant,5292,-14.9022,-17.7725,0.2541,0.0554,Not cycling,0.7228,Hypoxia,NaN,Right ovary


In [25]:
# Step 1: Merge directly on index
merged = ad.obs.merge(
    meta_data_df,
    left_index=True,
    right_index=True,
    how='left'
)

# Step 2: Assign back to AnnData object
ad.obs = merged
ad.obs

,doublet_score,predicted_doublet,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,Project_ID,Primary_or_Metastatic,sample,patient,...,complexity,umap1,umap2,g1s_score,g2m_score,cell_cycle_phase,mp_top_score,mp_top,mp_assignment,source
AAACCTGAGAGACGAA_VOA11543L,0.027233,False,4714,19148.0,959.0,5.008356,Zhang_2019,Primary,VOA11543L,VOA11543,...,4714,12.1482,27.5697,-0.1440,-0.0600,Not cycling,0.5393,CAF10,NaN,Left ovary
AAACCTGAGAGTCTGG_VOA11543L,0.014647,False,2103,6912.0,526.0,7.609954,Zhang_2019,Primary,VOA11543L,VOA11543,...,2103,29.2269,9.9902,0.0844,-0.0102,Not cycling,0.7347,Complement,NaN,Left ovary
AAACCTGCACAGGAGT_VOA11543L,0.040604,False,4820,20890.0,1275.0,6.103399,Zhang_2019,Primary,VOA11543L,VOA11543,...,4820,-24.3399,12.6990,-0.0118,0.0163,Not cycling,0.5252,Stress,NaN,Left ovary
AAACCTGGTACTTGAC_VOA11543L,0.056941,False,3452,12886.0,791.0,6.138445,Zhang_2019,Primary,VOA11543L,VOA11543,...,3452,25.6163,12.0930,NaN,NaN,NaN,NaN,NaN,NaN,Left ovary
AAACCTGTCGTTTAGG_VOA11543L,0.022640,False,3507,9733.0,409.0,4.202199,Zhang_2019,Primary,VOA11543L,VOA11543,...,3507,23.2708,-23.5292,0.0070,0.0145,Not cycling,1.4791,Endo1,Endo1,Left ovary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTCAATGGATA_VOA11543R,0.047067,False,2544,7782.0,511.0,6.566435,Zhang_2019,Primary,VOA11543R,VOA11543,...,2544,11.1967,26.8866,-0.1578,-0.0366,Not cycling,NaN,NaN,NaN,Right ovary
TTTGGTTGTCATTAGC_VOA11543R,0.079054,False,3608,19234.0,1436.0,7.465946,Zhang_2019,Primary,VOA11543R,VOA11543,...,3608,-10.5268,-13.0979,0.0691,0.0449,Not cycling,0.9034,Stress,NaN,Right ovary
TTTGGTTGTGTAACGG_VOA11543R,0.011727,False,2946,8983.0,531.0,5.911165,Zhang_2019,Primary,VOA11543R,VOA11543,...,2946,-5.3561,-15.7878,-0.1851,-0.0177,Not cycling,1.2055,Chromatin,NaN,Right ovary
TTTGGTTGTTCATGGT_VOA11543R,0.066787,False,4321,20494.0,1613.0,7.870596,Zhang_2019,Primary,VOA11543R,VOA11543,...,4321,-6.0200,-10.7464,0.1735,-0.0244,Not cycling,1.2051,Stress,Stress,Right ovary


In [26]:
ad.obs['cell_type'].value_counts()

cell_type
Malignant                 1351
Fibroblast                1162
Endothelial                306
Vascular_smooth_muscle     263
Macrophage                  95
T_cell                      22
B_cell                      15
Name: count, dtype: int64

In [27]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='cell_type', 
                          celltypes_to_keep=['Malignant'],
                          further_pre=True)
ad

Original shape: (3214, 24410)
Filtered shape: (1351, 24410)


/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1086: NumbaDeprecationWarning:

Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 1351 × 24410
    obs: 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'sample', 'patient', 'cell_type', 'complexity', 'umap1', 'umap2', 'g1s_score', 'g2m_score', 'cell_cycle_phase', 'mp_top_score', 'mp_top', 'mp_assignment', 'source'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'scrublet', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [ ]:
reset_plot()
sc.pl.umap(ad, color=['sample', 'source'])

In [32]:
# add more clinical information
ad.obs['Project_ID'] = 'Zhang_2019'
ad.obs['Primary_or_Metastatic'] = 'Primary'
ad.obs['Final_cancer_type'] = 'Ovarian Cancer'
ad.obs['Final_histological_subtype'] = 'Unknown'
ad.obs['Final_molecular_subtype'] = 'Unknown'
ad.obs['Final_tissue'] = 'Ovary'
ad.obs['Final_sample_id'] = ad.obs['sample']
ad.obs['Final_patient_age'] = 'Unknown'
ad.obs['Final_patient_stage'] = 'Unknown'
ad.obs['Final_patient_treatment'] = 'Naive'

In [33]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/Zhang_2019.OVC.h5ad', compression='gzip')

## Integrate the data

In [ ]:
data_dir = './Data/Cancer_cell_data/'
all_h5_files = os.listdir(data_dir)
all_h5_files.sort()

all_h5_files

In [ ]:
memory_usgae()

In [ ]:
from collections import defaultdict

cancer_ad_list = []

for h5 in all_h5_files:
    if 'ntegrated' in h5 or 'OVC' not in h5:
        continue

    print(h5)
    # continue
    ad = sc.read_h5ad(data_dir + h5)
    
    if h5.__contains__('2100'):
        ad.obs_names = ad.obs['Cell']
    # display(ad.obs)
    # continue
    # Fix .var_names
    if ad.var_names[0].startswith('ENSG'):
        new_names = [i.split('_')[0] for i in ad.var.feature_name]
    elif 'ENSG' in ad.var_names[0]:
        new_names = [i.split('_')[0] for i in ad.var_names]
    else:
        new_names = list(ad.var_names)

    # Assign new names
    ad.var_names = new_names
    ad.var_names_make_unique()

    # Fix raw.var names
    if ad.raw is not None:
        ad.raw._var.index = pd.Index(new_names).astype(str)
        # Ensure uniqueness
        seen = defaultdict(int)
        unique_names = []
        for name in ad.raw._var.index:
            if seen[name]:
                unique_names.append(f"{name}_{seen[name]}")
            else:
                unique_names.append(name)
            seen[name] += 1
        ad.raw._var.index = pd.Index(unique_names)

    # Clean obs + var
    # ad.obs = ad.obs.reset_index(drop=True)
    ad.obs_names_make_unique()
    ad.var_names_make_unique()
    
    ad.X = ad.raw.X.copy()  # copy ensures independence
    ad.raw = None           # delete .raw to save memory

    cancer_ad_list.append(ad)
    display(ad.to_df())
    # display(ad.raw.to_adata().to_df())
    # print(ad.raw.to_adata().to_df().max(axis=1))


In [ ]:
memory_usgae()

In [ ]:
for ad in cancer_ad_list:
    print(ad.raw.shape)

In [ ]:
memory_usgae()

In [ ]:
import sys
import numpy as np

total_cells = sum(ad.shape[0] for ad in cancer_ad_list)
total_genes = min(ad.shape[1] for ad in cancer_ad_list)
estimated_bytes = total_cells * total_genes * 4  # 4 bytes per float32
print(f"Estimated memory: {estimated_bytes / 1e9:.2f} GB")


In [ ]:
import psutil

mem = psutil.virtual_memory()
print(f"Available memory: {mem.available / 1e9:.2f} GB")
print(f"Total memory: {mem.total / 1e9:.2f} GB")


In [ ]:
from scipy.sparse import csr_matrix

# for ad in cancer_ad_list:
#    if not isinstance(ad.X, csr_matrix):
#        ad.X = csr_matrix(ad.X)


In [ ]:
combined_ad = ann.concat(cancer_ad_list, join="inner", axis=0)
combined_ad

In [ ]:
combined_ad.raw = ann.AnnData(
    X=combined_ad.X.copy(), 
    var=combined_ad.var.copy()
)

In [ ]:
combined_ad.raw.shape

In [ ]:
memory_usgae()

In [ ]:
combined_ad = reprocess_all(combined_ad)

In [ ]:
combined_ad.raw.shape

In [ ]:
combined_ad.obs["Final_histological_subtype"].value_counts()

In [ ]:
combined_ad.obs['Final_histological_subtype_backup'] = combined_ad.obs['Final_histological_subtype']

In [ ]:
def unify_ovarian_histology(val):
    val = str(val).strip().lower()
    if "mix" in val:
        return "OVC: Mixed high-grade serous + clear cell carcinoma"
    if "serous" in val:
        return "OVC: High-grade serous carcinoma"
    elif "endometrioid" in val:
        return "OVC: Endometrioid"
    else:
        return "OVC: Unspecified"

combined_ad.obs["Final_histological_subtype"] = combined_ad.obs["Final_histological_subtype_backup"].apply(unify_ovarian_histology)
combined_ad.obs["Final_histological_subtype"].value_counts()

In [ ]:
combined_ad.obs['Final_molecular_subtype'].value_counts()

In [ ]:
combined_ad.obs['Final_molecular_subtype'] = 'OVC: Unspecified'

In [ ]:
combined_ad.obs['Final_tissue'].value_counts()

In [ ]:
combined_ad.obs["Final_tissue"] = combined_ad.obs["Final_tissue"].str.replace('Adnexa', 'Ovary')

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

### Harmony integration

In [ ]:
combined_ad

In [ ]:
combined_ad.obs['Primary_or_Metastatic'].value_counts()

In [ ]:
memory_usgae()

In [ ]:
Z = harmonize(combined_ad.obsm['X_pca'], combined_ad.obs, batch_key = ['Project_ID'])


In [ ]:
combined_ad.obsm['X_pca_harmony'] = Z


In [ ]:
sc.pp.neighbors(combined_ad, n_neighbors=15, use_rep='X_pca_harmony')
sc.tl.umap(combined_ad)

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

In [ ]:
combined_ad.obs["Final_patient_age_backup"] = combined_ad.obs["Final_patient_age"]

def clean_patient_age(age):
    if pd.isna(age):
        return np.nan
    age = str(age).strip()
    if age.lower() == "unknown":
        return np.nan
    elif "-" in age:
        # Convert age ranges like '46-50' to their midpoint
        parts = age.split("-")
        try:
            return int((int(parts[0]) + int(parts[1])) / 2)
        except:
            return np.nan
    else:
        try:
            return int(age)
        except:
            return np.nan

# Apply cleaning
combined_ad.obs["Final_patient_age"] = combined_ad.obs["Final_patient_age_backup"].apply(clean_patient_age)


In [ ]:
combined_ad.obs["Final_patient_age_backup"] = combined_ad.obs["Final_patient_age_backup"].astype(str)

In [ ]:

combined_ad.write_h5ad('./Data/Cancer_cell_data/OVC_integrated.harmony.h5ad', compression='gzip')
